# Satellite Geolocation Solver: TDOA, FDOA & Error Ellipses Walkthrough

This notebook provides an end-to-end mathematical and computational walkthrough of satellite geolocation using:
1. **TDOA (Time Difference of Arrival)** and **FDOA (Frequency Difference of Arrival)** from Cross Ambiguity Function (CAF) measurements.
2. **CAF Peak Fidelity & State Covariance Propagation**: Translating frequency uncertainty (Hz), time uncertainty (s), and satellite ephemeris error covariances into target position confidence.
3. **WGS Ellipsoid Geodetic Modeling**: High-precision conversions and constraints using `pymap3d` (`wgs84`, `grs80`, etc.).
4. **Non-linear Least Squares (Gauss-Newton / Levenberg-Marquardt)**: Step-by-step mathematical derivation of TDOA isochrones, FDOA Doppler curves, and local ENU Jacobian transformations.
5. **Multiple Reference Emitters**: Transponder and downlink bias calibration using known stations.
6. **Error Ellipses & 3D Spheres/Ellipsoids**: Extraction of semi-major/minor axes, azimuth, and the quadratic equation constants ($Ax^2 + Bxy + Cy^2 = 1$).
7. **Alternative Geometries**: Sequences of TDOAs (multilateration) and sequences of FDOAs across multiple epochs or satellite passes.


## 1. Environment & Setup
We import standard mathematical packages, Skyfield, pymap3d, and our `sat_geo_solver` toolkit.


In [ ]:
from datetime import timedelta
import numpy as np
import matplotlib.pyplot as plt
import pymap3d as pm
from skyfield.api import load
from skyfield.iokit import parse_tle_file

from sat_geo_solver import (
    TwoSat,
    GeolocationSolver,
    ErrorEllipse,
    SatelliteState,
    Observation,
    plot_error_ellipse,
    plot_convergence_history,
    plot_fidelity_sensitivity,
)
print("Environment and sat_geo_solver successfully loaded.")


## 2. Satellite Orbit & Scenario Configuration
We load Two-Line Element (TLE) ephemerides for two geostationary satellites:
- Primary: **GOES-16** (`g16.tle`)
- Secondary: **Galaxy 19** (`g19.tle`)

We configure ground receivers in Colorado Springs, CO, a known reference transmitter (**PBS Network Operations Center** in Alexandria, VA), and a target unknown emitter (**Montana PBS** in Bozeman, MT).


In [ ]:
ts = load.timescale()
with open('g16.tle', 'rb') as f:
    sat_g16 = list(parse_tle_file(f, ts))[0]
with open('g19.tle', 'rb') as f:
    sat_g19 = list(parse_tle_file(f, ts))[0]

epoch_dt = sat_g16.epoch.utc_datetime()

# Ground antenna stations pointed at each satellite
cos_rx1 = [+38.4, -104.82]
cos_rx2 = [+38.4001, -104.8201]

# Reference transmitter (PBS NOC, Alexandria VA)
pbs_noc_ll = [+38.8622183, -77.0504056, 63]

# Ground Truth Target Emitter (Montana PBS, Bozeman MT)
montana_pbs_ll = [+45.6655208, -111.0514232, 271]

# RF Link configuration
translation_frequency = 2.3e9
pbs_downlink = 12.14e9
pbs_uplink = pbs_downlink + translation_frequency

# Instantiate TwoSat scenario
two_sat = TwoSat(sat_g16, sat_g19, epoch_dt, cos_rx1, cos_rx2)
print(f"Observation Epoch: {epoch_dt}")


## 3. Mathematical Foundations of TDOA & FDOA Geolocation

### Differential Time Offset (DTO) and TDOA
A signal emitted from position $\mathbf{x}$ arrives at Primary satellite $\mathbf{s}_1$ and Secondary satellite $\mathbf{s}_2$ before being downlinked to ground receivers $\mathbf{r}_1, \mathbf{r}_2$:
$$\tau_1(\mathbf{x}) = \frac{\|\mathbf{s}_1 - \mathbf{x}\| + \|\mathbf{r}_1 - \mathbf{s}_1\|}{c}$$
$$\tau_2(\mathbf{x}) = \frac{\|\mathbf{s}_2 - \mathbf{x}\| + \|\mathbf{r}_2 - \mathbf{s}_2\|}{c}$$
$$\text{DTO}(\mathbf{x}) = \tau_2(\mathbf{x}) - \tau_1(\mathbf{x})$$

When a reference emitter at known location $\mathbf{x}_{\text{ref}}$ is observed simultaneously, the common satellite clock, downlink delay, and transponder delay cancel out:
$$\text{TDOA} = \text{DTO}(\mathbf{x}_{\text{target}}) - \text{DTO}(\mathbf{x}_{\text{ref}}) = \frac{(\|\mathbf{s}_2 - \mathbf{x}\| - \|\mathbf{s}_1 - \mathbf{x}\|) - (\|\mathbf{s}_2 - \mathbf{x}_{\text{ref}}\| - \|\mathbf{s}_1 - \mathbf{x}_{\text{ref}}\|)}{c}$$

This defines a **hyperboloid of revolution** (isochrone) on the surface of the Earth.

### Differential Frequency Offset (DFO) and FDOA
Due to satellite relative motion $\mathbf{v}_1, \mathbf{v}_2$, the Doppler frequency shift is:
$$f_{\text{rx}, i} \approx f_0 \left(1 - \frac{\dot{r}_i}{c}\right), \quad \dot{r}_i = \frac{\mathbf{s}_i - \mathbf{x}}{\|\mathbf{s}_i - \mathbf{x}\|} \cdot \mathbf{v}_i$$

The differential frequency is:
$$\text{FDOA} = \text{DFO}(\mathbf{x}_{\text{target}}) - \text{DFO}(\mathbf{x}_{\text{ref}}) \approx -\frac{f_0}{c} \Big[ (\dot{r}_{2, \text{tar}} - \dot{r}_{1, \text{tar}}) - (\dot{r}_{2, \text{ref}} - \dot{r}_{1, \text{ref}}) \Big]$$

This defines an **isodoppler curve**. The intersection of the isochrone and the isodoppler on the Earth ellipsoid surface yields the 2D location solution.


## 4. Synthetic CAF Measurements
We simulate the CAF peak values that would be extracted by cross-correlating signals received from both satellites.


In [ ]:
dto_tar_true = float(two_sat.dto(montana_pbs_ll))
dto_ref_true = float(two_sat.dto(pbs_noc_ll))
tdoa_meas = dto_tar_true - dto_ref_true

dfo_tar_true = float(two_sat.dfo(montana_pbs_ll, pbs_uplink, translation_frequency))
dfo_ref_true = float(two_sat.dfo(pbs_noc_ll, pbs_uplink, translation_frequency))
fdoa_meas = dfo_tar_true - dfo_ref_true

print(f"Measured TDOA: {tdoa_meas*1e6:.4f} µs ({tdoa_meas:.9e} s)")
print(f"Measured FDOA: {fdoa_meas:.4f} Hz")


## 5. Non-linear Solver Execution (Gauss-Newton / Levenberg-Marquardt)
We now solve for the unknown emitter position using `GeolocationSolver`.
We supply:
- `initial_guess`: A coarse estimate (e.g. $[40.0^\circ, -100.0^\circ]$ in Nebraska/Kansas, hundreds of km away)
- `altitude`: Montana terrain elevation (271 m)
- `error_fidelity`: Peak resolution from the CAF: $\sigma_t = 1.0\,\mu\text{s}$, $\sigma_f = 0.5\,\text{Hz}$
- `max_iterations`, `tol`: Fully exposed iteration parameters


In [ ]:
# Initialize solver using high-accuracy WGS-84 model
solver = GeolocationSolver(ellipsoid='wgs84', max_iterations=50, tol=1e-4)

# Execute solver
result = solver.solve_from_twosat(
    two_sat=two_sat,
    tdoa=tdoa_meas,
    fdoa=fdoa_meas,
    uplink=pbs_uplink,
    translation_frequency=translation_frequency,
    ref_emitter=pbs_noc_ll,
    altitude=271.0,
    initial_guess=[40.0, -100.0],
    error_fidelity=(1.0e-6, 0.5), # (sigma_t, sigma_f)
    confidence=0.95,
)

print(result)


## 6. Tracing Mathematical Convergence
The solver iteratively adjusts the candidate position in local East-North coordinates:
$$\Delta \mathbf{x} = - (\mathbf{J}^T \mathbf{W} \mathbf{J} + \lambda \mathbf{I})^{-1} \mathbf{J}^T \mathbf{W} \mathbf{r}$$
Below is the iteration-by-iteration history table and convergence plot.


In [ ]:
print(f"{'Iter':<5} | {'Latitude':<11} | {'Longitude':<12} | {'Step (m)':<10} | {'Residual RMS':<12}")
print("-" * 60)
for step in result.history:
    print(f"{step.iteration:<5} | {step.lat:<11.6f} | {step.lon:<12.6f} | {step.step_m:<10.2f} | {step.residual_norm:<12.2e}")

# Plot convergence
fig, ax = plt.subplots(figsize=(8, 4))
plot_convergence_history(result, ax=ax)
plt.show()


## 7. Error Ellipse & Confidence Region Analysis

### Quadratic Equation Representation
The 2D confidence region in local horizontal meters (where $x = \text{East}$, $y = \text{North}$) satisfies:
$$\mathbf{r}^T \mathbf{C}_{\text{horiz}}^{-1} \mathbf{r} = k^2 \implies A x^2 + B x y + C y^2 = 1$$

where $k^2 = -2 \ln(1 - P)$ ($k^2 \approx 5.991$ for $95\%$ confidence).


In [ ]:
ee = result.error_ellipse
qc = ee.quadratic_constants

print("=== 95% Error Ellipse Parameters ===")
print(f"Semi-Major Axis (a) : {ee.semi_major_m:.2f} m")
print(f"Semi-Minor Axis (b) : {ee.semi_minor_m:.2f} m")
print(f"Azimuth (from North): {ee.azimuth_deg:.1f}°")
print(f"Polar Angle (from E): {ee.angle_deg:.1f}°")
print(f"Quadratic Form      : ({qc['A']:.4e})*x^2 + ({qc['B']:.4e})*x*y + ({qc['C']:.4e})*y^2 = 1")

# Verify boundary satisfaction
e_pts, n_pts = ee.to_enu_points(n_points=100)
test_vals = qc['A']*e_pts**2 + qc['B']*e_pts*n_pts + qc['C']*n_pts**2
print(f"Max numerical error from 1.0 along perimeter: {np.max(np.abs(test_vals - 1.0)):.2e}")


### Standard Operational Form and Confidence Scaling
In tactical GIS, radar, and navigation systems, error ellipses are expressed via their standard operational attributes:
- **Semi-Major Axis ($a$) and Semi-Minor Axis ($b$) in meters**
- **Azimuth bearing (degrees clockwise from North)**
- **Confidence Level ($P$) and Multiplier ($k = \sqrt{\chi^2_2(P)}$)**
- **Circular Error Probable equivalent ($\text{CEP}_{50}$ in meters)**

We also demonstrate **testing whether the error ellipse intersects a geographic zone or bounding box**.

In [ ]:
# 1. Operational Standard Form
std_dict = ee.standard_form
print("=== Operational Standard Form ===")
for k, v in std_dict.items():
    if k != "quadratic_constants":
        print(f"{k:<20}: {v}")

# 2. Zone Intersection Testing
# Define a test operational polygon around Bozeman MT in ENU offsets (meters)
test_zone_enu = [(-50000.0, -50000.0), (50000.0, -50000.0), (50000.0, 50000.0), (-50000.0, 50000.0)]
intersects = ee.intersects_zone(test_zone_enu)
print(f"Does the 95% confidence ellipse intersect the 100,000 m x 100,000 m region around center? {intersects}")

# Point containment test
print(f"Is the center point contained inside the ellipse? {ee.contains_point(0.0, 0.0)}")
print(f"Is an offset point at (10,000,000 m, 10,000,000 m) contained? {ee.contains_point(10000e3, 10000e3)}")


### Visualizing the Error Ellipse in Local ENU Space

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
plot_error_ellipse(result, ax=ax, true_pos=montana_pbs_ll)
plt.show()


## 8. CAF Peak Fidelity Sensitivity Analysis
How do uncertainties in frequency ($\sigma_f$ in Hz) and time ($\sigma_t$ in µs) affect the geometry of the error ellipse?
We sweep both fidelity domains below.


In [ ]:
sigma_t_sweep = np.linspace(0.01e-6, 0.25e-6, 10) # 0.1 to 2.0 microseconds
sigma_f_sweep = np.linspace(0.01, 0.1, 10)        # 0.05 to 1.0 Hz

fig, (ax1, ax2) = plot_fidelity_sensitivity(
    two_sat=two_sat,
    target_pos=montana_pbs_ll,
    ref_pos=pbs_noc_ll,
    uplink=pbs_uplink,
    translation_frequency=translation_frequency,
    sigma_t_range=sigma_t_sweep,
    sigma_f_range=sigma_f_sweep,
    altitude=271.0,
    confidence=0.95
)
plt.show()


## 9. Sequences of TDOA Only (Multilateration) & FDOA Only

When frequency fidelity is low or unavailable, can a sequence of TDOAs over time solve for the emitter?
Yes! As satellites move across their orbital figure-eight / diurnal paths, the geometry changes, creating intersecting isochrones over time.


In [ ]:
# Generate multi-epoch scenario (12 epochs spaced by 20 minutes)
dt_multi = [epoch_dt + timedelta(minutes=20*i) for i in range(12)]
two_sat_multi = TwoSat(sat_g16, sat_g19, dt_multi, cos_rx1, cos_rx2)

# Compute TDOA sequence
dto_tar_seq = two_sat_multi.dto(montana_pbs_ll)
dto_ref_seq = two_sat_multi.dto(pbs_noc_ll)
tdoa_sequence = dto_tar_seq - dto_ref_seq

# Solve using TDOA sequence ONLY (no FDOA)
res_tdoa_only = solver.solve_from_twosat(
    two_sat=two_sat_multi,
    tdoa=tdoa_sequence,
    fdoa=None, # TDOA-only!
    ref_emitter=pbs_noc_ll,
    altitude=271.0,
    initial_guess=[43.0, -108.0],
)

print("=== TDOA-Only Multilateration Result ===")
print(res_tdoa_only)
true_ecef = np.array(pm.geodetic2ecef(*montana_pbs_ll))
dist_err = float(np.linalg.norm(true_ecef - res_tdoa_only.ecef))
print(f"Target position error: {dist_err:.2f} meters")


## 10. Multi-Reference Emitter Calibration
When multiple reference emitters are available (e.g. PBS NOC in VA and a second ground station in Denver, CO), the solver combines the differential calibration measurements to reduce transponder bias uncertainty:
$$\bar{\beta} = \frac{\sum w_i \beta_i}{\sum w_i}, \quad \sigma_{\bar{\beta}}^2 = \frac{1}{\sum w_i}$$


In [ ]:
ref1 = pbs_noc_ll
ref2 = [39.7392, -104.9903, 1609.0] # Denver, CO

dto_ref1 = float(two_sat.dto(ref1))
dto_ref2 = float(two_sat.dto(ref2))
dto_ref_avg = (dto_ref1 + dto_ref2) / 2.0

dfo_ref1 = float(two_sat.dfo(ref1, pbs_uplink, translation_frequency))
dfo_ref2 = float(two_sat.dfo(ref2, pbs_uplink, translation_frequency))
dfo_ref_avg = (dfo_ref1 + dfo_ref2) / 2.0

tdoa_cal = dto_tar_true - dto_ref_avg
fdoa_cal = dfo_tar_true - dfo_ref_avg

res_multi_ref = solver.solve_from_twosat(
    two_sat=two_sat,
    tdoa=tdoa_cal,
    fdoa=fdoa_cal,
    uplink=pbs_uplink,
    translation_frequency=translation_frequency,
    ref_emitter=[ref1, ref2],
    altitude=271.0,
    initial_guess=[44.0, -110.0],
)

print("=== Multi-Reference Solution ===")
print(f"Estimated Lat: {res_multi_ref.lat:.6f}°, Lon: {res_multi_ref.lon:.6f}°")
print(f"True Lat:      {montana_pbs_ll[0]:.6f}°, Lon: {montana_pbs_ll[1]:.6f}°")


## 11. Altitude Support: Constrained Surface vs 3D Error Ellipsoid / Sphere

### Does Altitude Require More Than One Reference?
**No, altitude does not mathematically require additional reference stations.**
- A single reference emitter is sufficient to calibrate satellite transponder time and frequency bias.
- However, solving for altitude as a free 3D variable requires at least **3 independent geometric measurements** (or constraining altitude to a known Digital Elevation Model / terrain height, which reduces the problem to 2 horizontal dimensions).
- When altitude is unknown and solved in 3D, the vertical uncertainty is characterized by a **3D Error Ellipsoid**:
  $$\mathbf{r}^T \mathbf{C}_{3D}^{-1} \mathbf{r} = k_{3D}^2$$
  and its spherical equivalent is the **Spherical Error Probable (SEP)**:
  $$\text{SEP} \approx 0.51 (\sigma_x + \sigma_y + \sigma_z)$$


In [ ]:
# Example: 3D Unconstrained Error Ellipsoid Demonstration
cov_3d = np.array([
    [2.5e7, 1.2e6, 4.0e5],
    [1.2e6, 8.0e5, 2.0e5],
    [4.0e5, 2.0e5, 5.0e6]
])

from sat_geo_solver.models import ErrorEllipsoid
ellipsoid_3d = ErrorEllipsoid.from_covariance_3d(cov_3d, confidence=0.95)

print("=== 3D Error Ellipsoid & Sphere Parameters ===")
print(f"Semi-axes (a, b, c)        : {ellipsoid_3d.semi_axes_m[0]:.2f} m, {ellipsoid_3d.semi_axes_m[1]:.2f} m, {ellipsoid_3d.semi_axes_m[2]:.2f} m")
print(f"Spherical Error Probable (SEP): {ellipsoid_3d.sep_m:.2f} m")


## 12. Searching for Unknown Emitter Locations ()

When the emitter location is completely unknown and no initial guess is available, setting  scans a coarse 2D grid across the mutual satellite coverage area to find the global cost minimum basin before refining with Levenberg-Marquardt optimization.

In [ ]:
# Execute solver for an unknown emitter without any initial guess!
res_unknown = solver.solve_from_twosat(
    two_sat=two_sat,
    tdoa=tdoa_meas,
    fdoa=fdoa_meas,
    uplink=pbs_uplink,
    translation_frequency=translation_frequency,
    ref_emitter=pbs_noc_ll,
    altitude=271.0,
    initial_guess=None,          # No operator initial guess provided!
    coarse_grid_search=True,     # Automatically search coarse grid first
    coarse_grid_bounds=(35.0, 55.0, -125.0, -95.0), # Regional search bounds
)

print("=== Unknown Emitter Solution (via coarse_grid_search) ===")
print(res_unknown)
true_ecef = np.array(pm.geodetic2ecef(*montana_pbs_ll))
dist_err = float(np.linalg.norm(true_ecef - res_unknown.ecef))
print(f"Recovered location error: {dist_err:.2f} meters")
